# 05 - Staking Mechanics Verification

This notebook validates staking-mechanics claims against contract-source check artifacts,
with optional live RPC reads for selected contract state values.

**Source classes:** ONCHAIN_EXECUTED-derived checks + optional live RPC queries.


## Verification Scope

- Transfer restrictions for `stSUMR`
- Presence of lockup/penalty constants in `SummerStaking`
- Optional live reads (`totalSupply`, `paused`, `name`, `symbol`) for `stSUMR`


In [ ]:
from pathlib import Path
import sys
import json
import warnings

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

sns.set_theme(style="whitegrid")

cwd = Path.cwd().resolve()
root = None
for candidate in [cwd, *cwd.parents]:
    if (candidate / "notebooks" / "notebook_utils.py").exists():
        root = candidate
        break
if root is None:
    raise RuntimeError("Could not locate repository root containing notebooks/notebook_utils.py")

if root.as_posix() not in sys.path:
    sys.path.insert(0, root.as_posix())

from notebooks import notebook_utils as nbx
ROOT = nbx.setup_notebook_context()
print(f"Project root: {ROOT}")
print(f"Notebook run started (UTC): {nbx.utc_now_iso()}")


In [ ]:
from src.config import BASE_RPC_URL, ST_SUMR
from web3 import Web3

RUN_LIVE_CONTRACT_CALLS = False
EVIDENCE_DIR = nbx.latest_evidence_dir()
if EVIDENCE_DIR is None:
    raise FileNotFoundError("No evidence directory found.")

checks = nbx.load_json(EVIDENCE_DIR / "contract_source_checks.json")
checks


In [ ]:
expected_true_fields = [
    "stsumr_has_canTransfer",
    "stsumr_revert_transfer_not_allowed",
    "stsumr_mint_or_burn_only_rule",
    "summerstaking_has_MAX_LOCKUP_PERIOD",
    "summerstaking_has_MIN_PENALTY_2pct",
    "summerstaking_has_MAX_PENALTY_20pct",
    "summerstaking_has_FIXED_PENALTY_110d",
    "summerstaking_has_WEIGHTED_COEFF_700",
]

check_rows = []
for field in expected_true_fields:
    value = bool(checks.get(field))
    check_rows.append({"check": field, "value": value, "passes": value is True})

check_df = pd.DataFrame(check_rows)
display(check_df)

pass_rate = check_df["passes"].mean() if not check_df.empty else np.nan
display(Markdown(f"**Contract-source check pass rate:** `{pass_rate * 100:.1f}%`"))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.2), constrained_layout=True)
colors = ["#2a9d8f" if p else "#d62828" for p in check_df["passes"]]
ax.barh(check_df["check"], check_df["passes"].astype(int), color=colors)
ax.set_xlim(0, 1)
ax.set_xlabel("Pass (1=yes)")
ax.set_title("Staking Mechanics Checks")
plt.show()


In [ ]:
live_rows = []
if RUN_LIVE_CONTRACT_CALLS:
    if not BASE_RPC_URL:
        raise RuntimeError("BASE_RPC_URL is required for live contract calls.")

    w3 = Web3(Web3.HTTPProvider(BASE_RPC_URL, request_kwargs={"timeout": 30}))
    if not w3.is_connected():
        raise ConnectionError("Could not connect to Base RPC endpoint.")

    abi = [
        {"name": "name", "type": "function", "stateMutability": "view", "inputs": [], "outputs": [{"type": "string"}]},
        {"name": "symbol", "type": "function", "stateMutability": "view", "inputs": [], "outputs": [{"type": "string"}]},
        {"name": "totalSupply", "type": "function", "stateMutability": "view", "inputs": [], "outputs": [{"type": "uint256"}]},
        {"name": "decimals", "type": "function", "stateMutability": "view", "inputs": [], "outputs": [{"type": "uint8"}]},
        {"name": "paused", "type": "function", "stateMutability": "view", "inputs": [], "outputs": [{"type": "bool"}]},
    ]
    contract = w3.eth.contract(address=Web3.to_checksum_address(ST_SUMR), abi=abi)
    decimals = int(contract.functions.decimals().call())
    total_supply_raw = int(contract.functions.totalSupply().call())

    live_rows.append(
        {
            "name": contract.functions.name().call(),
            "symbol": contract.functions.symbol().call(),
            "paused": bool(contract.functions.paused().call()),
            "total_supply_tokens": total_supply_raw / (10 ** decimals),
            "block_number": int(w3.eth.block_number),
        }
    )

if live_rows:
    display(pd.DataFrame(live_rows))
else:
    display(Markdown("Live RPC reads skipped (`RUN_LIVE_CONTRACT_CALLS=False`)."))
